## Task 06 | Banking Ledger with OOP

In [1]:
from collections import namedtuple

# Immutable transaction record (banne ke baad modify nahi ho sakta)
Transaction = namedtuple("Transaction", ["transaction_id", "action", "amount", "resulting_balance"])


class Account:
    def __init__(self, account_id, initial_balance=0.0):
        if not account_id or type(account_id) != str:
            raise ValueError("Account ID valid string honi chahiye")

        if type(initial_balance) not in (int, float) or initial_balance < 0:
            raise ValueError("Initial balance negative nahi ho sakta")

        self._account_id = account_id
        self._balance = float(initial_balance)
        self._transactions = []
        self._tx_counter = 0

        # Agar shuru mein kuch balance hai to opening record add karein
        if self._balance > 0:
            self._record_transaction("OPENING", self._balance)

    @property
    def account_id(self):
        return self._account_id

    @property
    def balance(self):
        return self._balance

    def _validate_amount(self, amount):
        # Bool ko number samajhne se rokne ke liye explicit type check
        if type(amount) not in (int, float) or amount <= 0:
            raise ValueError("Amount positive number hona chahiye")

    def _record_transaction(self, action, amount):
        self._tx_counter += 1
        tx_id = f"{self._account_id}-TX{self._tx_counter:04d}"
        record = Transaction(tx_id, action, float(amount), self._balance)
        self._transactions.append(record)
        return record

    def deposit(self, amount):
        self._validate_amount(amount)
        self._balance += float(amount)
        return self._record_transaction("DEPOSIT", amount)

    def withdraw(self, amount):
        self._validate_amount(amount)
        if amount > self._balance:
            raise ValueError("Insufficient funds: account mein itna balance nahi hai")
        self._balance -= float(amount)
        return self._record_transaction("WITHDRAWAL", amount)

    def get_statement(self):
        # Original list ki copy return karein taake history safe rahe
        return list(self._transactions)


class Bank:
    def __init__(self):
        self._accounts = {}

    def create_account(self, account_id, initial_balance=0.0):
        if account_id in self._accounts:
            raise ValueError(f"Account '{account_id}' pehle se maujood hai")
        acc = Account(account_id, initial_balance)
        self._accounts[account_id] = acc
        return acc

    def get_account(self, account_id):
        if account_id not in self._accounts:
            raise ValueError(f"Account '{account_id}' maujood nahi hai")
        return self._accounts[account_id]

    def deposit(self, account_id, amount):
        acc = self.get_account(account_id)
        return acc.deposit(amount)

    def withdraw(self, account_id, amount):
        acc = self.get_account(account_id)
        return acc.withdraw(amount)

    def transfer(self, from_id, to_id, amount):
        # Pehle dono accounts aur amount validate karein
        sender = self.get_account(from_id)
        receiver = self.get_account(to_id)

        if from_id == to_id:
            raise ValueError("Same account mein transfer nahi ho sakta")

        if type(amount) not in (int, float) or amount <= 0:
            raise ValueError("Transfer amount positive hona chahiye")

        if amount > sender.balance:
            raise ValueError("Insufficient funds: sender ke paas balance kam hai")

        # Atomic operation: pehle sender se nikalein, phir receiver mein daalein
        old_sender_balance = sender._balance
        old_sender_tx_len = len(sender._transactions)
        old_sender_counter = sender._tx_counter

        try:
            sender._balance -= float(amount)
            sender._record_transaction("TRANSFER_OUT", amount)

            receiver._balance += float(amount)
            receiver._record_transaction("TRANSFER_IN", amount)
        except Exception:
            # Agar kisi bhi wajah se error aaye to sender ko wapas purani state par le aayein
            sender._balance = old_sender_balance
            sender._transactions = sender._transactions[:old_sender_tx_len]
            sender._tx_counter = old_sender_counter
            raise

    def get_statement(self, account_id):
        acc = self.get_account(account_id)
        return acc.get_statement()


# Testing
bank = Bank()
acc_a = bank.create_account("A", 1000)
acc_b = bank.create_account("B", 200)

bank.transfer("A", "B", 350)
print("A Balance:", acc_a.balance)  # 650.0
print("B Balance:", acc_b.balance)  # 550.0

print("\nStatement for A:")
for tx in bank.get_statement("A"):
    print(tx)

A Balance: 650.0
B Balance: 550.0

Statement for A:
Transaction(transaction_id='A-TX0001', action='OPENING', amount=1000.0, resulting_balance=1000.0)
Transaction(transaction_id='A-TX0002', action='TRANSFER_OUT', amount=350.0, resulting_balance=650.0)
